In [13]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, udf, to_timestamp
from pyspark.sql.types import *
import os
from pathlib import Path

# Determine the base directory of the script
try:
    base_dir = Path(__file__).resolve().parent
except NameError:
    base_dir = Path.cwd()
 
# Set HADOOP_HOME and update PATH
hadoop_home = "C:\\vs_code_projects\\python\\hadoop-3.2.1"
os.environ['HADOOP_HOME'] = str(hadoop_home)
os.environ['PATH'] += os.pathsep + str(hadoop_home + '\\bin')

# Create Spark session
print('starting session')
spark = SparkSession. \
        builder. \
        appName("NYC Taxi Pipeline"). \
        config("spark.executor.cores", 1). \
        config("spark.executor.instances", 1). \
        config("spark.executor.memory", "1g"). \
        config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1"). \
        config("spark.hadoop.io.native.lib.available", "false"). \
        master("local[*]"). \
        getOrCreate()
print('session created')

starting session
session created


In [14]:
def read_csv(file_path, schema):
    """Read the taxi data from a CSV file"""
    return spark.read.csv(file_path, schema=schema, header=True)

In [15]:
def read_parquet(file_path):
    """Read a single parquet file"""
    return spark.read.parquet(file_path)

In [16]:
taxi01_df = read_parquet(r"..\data\input_data\taxi_17_01.parquet")

In [17]:
taxi02_df = read_parquet(r"..\data\input_data\taxi_17_02.parquet")

In [18]:
zone_schema = StructType([StructField('LocationID', IntegerType(), True), 
                          StructField('Borough', StringType(), True), 
                          StructField('Zone', StringType(), True), StructField('service_zone', StringType(), True)])
zone_df = read_csv(r"..\data\input_data\taxi_zone_lookup.csv", schema = zone_schema)

In [19]:
taxi01_df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2017-01-01 00:32:05|  2017-01-01 00:37:48|              1|          1.2|         1|                 N|         140|         236|           2|        6.5|  0.5|    0.5|       0.

In [20]:
taxi02_df.show(4)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2017-02-01 00:19:20|  2017-02-01 00:25:56|              1|          2.9|         1|                 N|          75|         162|           2|        9.5|  0.5|    0.5|       0.

In [21]:
zone_df.show(4)

+----------+---------+--------------------+------------+
|LocationID|  Borough|                Zone|service_zone|
+----------+---------+--------------------+------------+
|         1|      EWR|      Newark Airport|         EWR|
|         2|   Queens|         Jamaica Bay|   Boro Zone|
|         3|    Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|Manhattan|       Alphabet City| Yellow Zone|
+----------+---------+--------------------+------------+
only showing top 4 rows


In [22]:
print(f"Total records in zone data: {zone_df.count()}")
print(f"Total records in 2015 data: {taxi01_df.count()}")
print(f"Total records in 2016 data: {taxi02_df.count()}")



Total records in zone data: 265
Total records in 2015 data: 9710820
Total records in 2016 data: 9169775


In [23]:
taxi01_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\taxi01")

In [24]:
taxi02_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\taxi02")

In [25]:
zone_df.write.mode("overwrite").parquet("C:\\vs_code_projects\\projects\\nyc_taxi_trips\\data\\output_data\\01_load_data\\zone")


In [26]:
spark.stop()